# POS Terminal Explorer

Pulls one week of July transactions to surface all distinct POS terminal IDs (`rnm`) and their current store name mappings.
Once you see the full IDs for terminals ending in **6458** (Neptune) and **2898** (Kosa), add them to `pipeline/config.py`.

In [ ]:
# ── Date Range Config ──────────────────────────────────────────────
# One week in July — adjust as needed

DATE_FROM = "2025-07-01"
DATE_TO   = "2025-07-07"

print(f"Sample window: {DATE_FROM} → {DATE_TO}")

In [ ]:
from pyspark.sql import functions as F
from pipeline.config import TRANSACTIONS_TABLE

df = spark.table(TRANSACTIONS_TABLE).filter(
    (F.col("date") >= DATE_FROM) &
    (F.col("date") <= DATE_TO)
).toPandas()

print(f"{len(df):,} rows loaded")
df.head(2)

In [ ]:
# ── All distinct POS terminals in this window ───────────────────────
import pandas as pd

# Extra names for terminals not yet in config.py
EXTRA_NAMES = {
    "6458": "НЕПТУН",
    "2898": "КОСА",
}

terminals = (
    df.groupby(["rnm", "store_name"])
      .agg(orders=("order_number", "nunique"), rows=("rnm", "count"))
      .reset_index()
      .sort_values("rows", ascending=False)
)

# Annotate with extra names where suffix matches
def resolve_name(row):
    suffix = str(row["rnm"])[-4:]
    if suffix in EXTRA_NAMES and "UNKNOWN" in str(row["store_name"]):
        return EXTRA_NAMES[suffix]
    return row["store_name"]

terminals["resolved_name"] = terminals.apply(resolve_name, axis=1)

print(f"{len(terminals)} unique POS terminals found")
terminals[["rnm", "resolved_name", "orders", "rows"]]

In [ ]:
# ── Daily sales by store (with resolved names) ──────────────────────

suffix_map = {f"UNKNOWN_{k}": v for k, v in EXTRA_NAMES.items()}
df["store_display"] = df["store_name"].replace(suffix_map)

daily = (
    df[~df["is_return"] & ~df["is_topping"]]
      .groupby(["date", "store_display"])
      .agg(revenue=("revenue", "sum"), orders=("order_number", "nunique"))
      .reset_index()
      .sort_values(["date", "store_display"])
)

daily

In [ ]:
# ── Config snippet to copy into pipeline/config.py ──────────────────
# Run the cell above first — copy the full rnm values from the terminals table,
# then paste the two new entries below into POS_TO_STORE in config.py.

for _, row in terminals[terminals["resolved_name"].isin(["НЕПТУН", "КОСА"])].iterrows():
    print(f'    "{row["rnm"]}": "{row["resolved_name"]}",  # add to POS_TO_STORE')